# LC 78 — Subsets
**Day 39 | Theme: Backtracking | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">

**Core Insight:** At every node in the recursion tree, the current
path *is already* a valid subset. Collect it immediately, then
extend by looping forward — never backward — to avoid duplicates.

</div>

## Official Problem Statement

Given an integer array `nums` of **unique** elements, return
*all possible subsets (the power set)*.

The solution set **must not** contain duplicate subsets.
Return the solution in **any order**.

**Constraints:**
- `1 <= nums.length <= 10`
- `-10 <= nums[i] <= 10`
- All integers in `nums` are **unique**.

## What This Is Actually Asking

For each element you have two choices: include it or skip it.
Doing this for every element independently produces 2^n combinations.
The empty list `[]` and the full list are both valid subsets.
All elements are unique, so no de-duplication logic is needed.
You need every subset, not just ones meeting some sum condition.

## Walk Through an Example by Hand

`nums = [1, 2, 3]`

```
Call backtrack(start=0, path=[])
  -> collect []                        # empty subset
  i=0: path=[1]
    Call backtrack(start=1, path=[1])
      -> collect [1]
      i=1: path=[1,2]
        Call backtrack(start=2, path=[1,2])
          -> collect [1,2]
          i=2: path=[1,2,3]
            Call backtrack(start=3, path=[1,2,3])
              -> collect [1,2,3]       # loop doesn't run
          pop -> path=[1,2]
        pop -> path=[1]
      i=2: path=[1,3]
        Call backtrack(start=3, path=[1,3])
          -> collect [1,3]
        pop -> path=[1]
    pop -> path=[]
  i=1: path=[2] ... (collect [2],[2,3])
  i=2: path=[3] ... (collect [3])
```

Final: `[[], [1], [1,2], [1,2,3], [1,3], [2], [2,3], [3]]`

## The Picture

```
                    []
           /        |        \
         [1]       [2]       [3]
        /   \       |
     [1,2] [1,3]  [2,3]
       |
   [1,2,3]
```

**Rule:** from index `i`, the next level can only pick
indices `>= i+1`.  This prevents `[1,2]` and `[2,1]`
from both appearing.

```
collect path[:] at EVERY node (not just leaves)
          ^
          This is the key difference from permutations.
```

## When To Use This Pattern

- When the problem says "all subsets" or "power set", think
  backtrack-and-collect-at-every-node.
- When elements are unique and order does not matter, think
  forward-only loop (`start` index).
- When you need *all combinations of size k*, think the same
  pattern but only collect when `len(path) == k`.
- When the answer is exponential in size (2^n), think backtracking
  rather than dynamic programming.
- When duplicate elements appear, think sort + skip-duplicate
  guard (LC 90 variant).

## The Approach

Maintain a running `path` list and a `start` index.
At the top of each recursive call, snapshot `path[:]` into
the result — this captures the subset at that tree node.
Then loop from `start` to `len(nums)`, appending `nums[i]`,
recursing with `i+1`, and popping to restore state.
Because we always move forward, each subset is built
exactly once without any visited set.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """Order-independent subset equality check."""
    def norm(result):
        return sorted(tuple(sorted(s)) for s in result)

    cases = [
        # (input, expected)
        ([1, 2, 3],
         [[], [1], [2], [3], [1,2], [1,3], [2,3], [1,2,3]]),
        ([0],
         [[], [0]]),
        ([-1, 0],
         [[], [-1], [0], [-1, 0]]),
        ([1],
         [[], [1]]),
        ([1, 2],
         [[], [1], [2], [1, 2]]),
    ]

    passed = 0
    for i, (nums, expected) in enumerate(cases):
        res = func(nums)
        if norm(res) == norm(expected):
            print(f"  Case {i+1}: PASSED")
            passed += 1
        else:
            print(f"  Case {i+1}: FAILED")
            print(f"    Input   : {nums}")
            print(f"    Expected: {norm(expected)}")
            print(f"    Got     : {norm(res)}")

    print(f"\n  Summary: {passed}/{len(cases)} passed")

In [ ]:
def subsets(nums: List[int]) -> List[List[int]]:
    """
    Return all subsets of nums (the power set).

    Strategy: backtracking.
      - Collect path[:] at EVERY recursive call (not just leaves).
      - Loop from `start` index to avoid re-using earlier elements.
      - Append element, recurse with i+1, then pop (undo).

    Args:
        nums: list of unique integers

    Returns:
        All 2^n subsets in any order.

    Time : O(n * 2^n)  — 2^n subsets, each copied in O(n)
    Space: O(n)        — recursion depth + path buffer
    """
    result = []
    path = []

    def backtrack(start: int) -> None:
        # --- debug: see each node as it is visited ---
        print(f"  [debug] collect path={path}, start={start}")
        result.append(path[:])          # collect current subset

        for i in range(start, len(nums)):
            path.append(nums[i])
            print(f"  [debug] chose nums[{i}]={nums[i]},"
                  f" path now={path}")
            backtrack(i + 1)            # move forward only
            popped = path.pop()
            print(f"  [debug] backtrack: popped {popped},"
                  f" path={path}")

    backtrack(0)
    return result


# Quick smoke test
print("subsets([1,2,3]):")
print(subsets([1, 2, 3]))

In [ ]:
# Uncomment and run when solution is ready
# test_harness(subsets)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (bitmask) | O(n · 2^n) | O(n · 2^n) | Loop 0..2^n, decode bits |
| Backtracking (optimal) | O(n · 2^n) | O(n) | Path buffer + call stack |

- Both approaches are the same *time* complexity because you must
  output 2^n subsets regardless.
- Backtracking wins on *auxiliary* space: the path is reused
  in-place rather than building new lists at each step.
- Output storage O(n · 2^n) is unavoidable (you must return
  every subset).

## Real World Connection

At **Citi**, risk stress-testing requires running every possible
combination of market shocks across a small set of risk factors
(interest rate, FX, credit spread, equity) — exactly the power
set pattern.
On **AWS Glue**, deciding which optional transformations to apply
to a dataset pipeline (drop nulls, cast types, rename columns)
can be modelled as subset enumeration when evaluating job
configurations.
In **data engineering**, feature selection for ML pipelines
often requires iterating over subsets of candidate features to
find the best performing combination.
Understanding 2^n growth early prevents you from accidentally
designing systems that enumerate subsets of large sets in
production.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra